#Drive Mount To Access Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/My Drive/CS412') #change if you need
!ls

# Load Data From Json Files



In [ ]:
import json
import pandas as pd
import random

def load_review_data_reservoir(json_file_path, sample_size):
  '''
    Load sampled data from the full dataset of 7 million reviews.

    Args:  json_file_path(string): path to load data from
           sample_size (int): how many samples to load

    Returns: DataFrame for sampled data
  '''
  data_reservoir = []

  with open(json_file_path, "r", encoding="utf-8") as review_json_file:
    for i, row in enumerate(review_json_file):
      review = json.loads(row)

      if len(data_reservoir) < sample_size:
        data_reservoir.append(review)
      else:
        random_position = random.randint(0, i)
        if random_position < sample_size:
          data_reservoir[random_position] = review

  review_df = pd.DataFrame(data_reservoir)
  review_df.to_json("review_sampled.json", orient="records", lines=True)

  return review_df

review_df = load_review_data_reservoir("yelp_academic_dataset_review.json", 200000)

In [ ]:
#load full data from json_file_path
def load_data(json_file_path):
  '''
    Load sampled data from the full dataset of 7 million reviews.

    Args:  json_file_path(string): path to load data from
           sample_size (int): how many samples to load

    Returns: DataFrame for sampled data
  '''
  data_chunks = pd.read_json(json_file_path, lines=True, chunksize=1000000)
  data_frame = pd.concat(data_chunks)
  return data_frame

# Data Exploration

In [ ]:
#data Exploration

def data_shape_features(data_frame):
  '''
    Print out Dataset shape and features, as well as head and tail of data.

    Args:  data_frame(DataFrame): dataframe to explore
  '''
  print("Data Set Shape: ", data_frame.shape)
  print("\n")
  print("Data Features: ", data_frame.columns)
  print("\n")
  print("Data Features Head: ")
  display(data_frame.head(5))
  print("\n")
  print("Data Features Tail: ")
  display(data_frame.head(5))
  print("\n")

def data_basic_info(data_frame):
  '''
    Print out Dataset info.

    Args:  data_frame(DataFrame): dataframe to explore
  '''
  print("Data Info: ")
  data_frame.info(show_counts=True)
  print("\n")

def min_max_numeric_cols(data_frame):
  '''
    Print out Dataset min and max of numeric columns.

    Args:  data_frame(DataFrame): dataframe to explore
  '''
  print("Data Min and Max of Numeric Columns: ")
  print(data_frame.select_dtypes(include=['int64', 'float64']).agg(['min', 'max']))
  print("\n")

def stars_values(data_frame):
  '''
    Print out Dataset target 'stars' values and how many reviews are in each star category.

    Args:  data_frame(DataFrame): dataframe to explore
  '''
  print("Stars Unique Values: ", data_frame['stars'].unique())
  print("\n")
  print("Stars Unique Values Count: ", data_frame['stars'].value_counts())
  print("\n")

Review Data Exploration

In [ ]:
data_shape_features(review_df)
data_basic_info(review_df)
min_max_numeric_cols(review_df)
stars_values(review_df)

del review_df

User Data Exploration

In [ ]:
user_df = load_data("yelp_academic_dataset_user.json")

data_shape_features(user_df)
data_basic_info(user_df)
min_max_numeric_cols(user_df)

del user_df

Business Data Exploration

In [ ]:
business_df = load_data("yelp_academic_dataset_business.json")

data_shape_features(business_df)
data_basic_info(business_df)
min_max_numeric_cols(business_df)
stars_values(business_df)

del business_df

#Clean Data, Remove Gibberish, and Create A Sample Set

In [ ]:
!pip install gibberish-detector

In [ ]:
from gibberish_detector import detector
import concurrent.futures

def gibberish_detection_model(model_file_path):
  '''
    Load gibberish detection model from file and create detector.

    Args:  model_file_path(string): path to get training text from

    Returns: Gibberish detector object
  '''
  gibberish_detector = detector.create_from_model(model_file_path)
  return gibberish_detector

def is_gibberish(text):
  return gibberish_detector.is_gibberish(text)

def clean_text_col_review_df(review_text):
  '''
    Basic cleaning of data, removing excess whitespace.

    Args:  review_text(string): review to clean

    Returns: string, cleaned text
  '''
  review_text = review_text.replace("\n", " ")
  review_text = review_text.replace("\r", " ")
  review_text = review_text.strip()
  review_text = " ".join(review_text.split())

  return review_text

def load_and_clean_data(json_file_path, gibberish_detector):
  '''
    Load data from json_file_path, and create two other json files.

    One without gibberish filtering and one with.

    Args:  json_file_path(string): path to load data from
           gibberish_detector: gibberish detector model
  '''
  data_with_gibberish_filter = []
  data_without_gibberish_filter = []

  with open(json_file_path, "r", encoding="utf-8") as review_json_file:
    for i, row in enumerate(review_json_file):
      review = json.loads(row)
      clean_text_col_review_df(review["text"])

      data_without_gibberish_filter.append(review)

      is_gibberish = gibberish_detector.is_gibberish(review["text"])

      if is_gibberish:
        continue

      data_with_gibberish_filter.append(review)

  review_with_gibberish_filter_df = pd.DataFrame(data_with_gibberish_filter)
  review_with_gibberish_filter_df.to_json("data_with_gibberish_filter.json", orient="records", lines=True)
  del review_with_gibberish_filter_df

  review_without_gibberish_filter_df = pd.DataFrame(data_without_gibberish_filter)
  review_without_gibberish_filter_df.to_json("data_without_gibberish_filter.json", orient="records", lines=True)
  del review_without_gibberish_filter_df

  print("Review Data Set Length After Gibberish Filtering: ", len(data_with_gibberish_filter))
  print("Review Data Set Length Without Gibberish Filtering: ", len(data_without_gibberish_filter))

gibberish_detector = gibberish_detection_model("big.model")
load_and_clean_data("review_sampled.json", gibberish_detector)

#Sentiment Analysis and Feature Building

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import pipeline

def sentiment_analysis(df):
  '''
    Run the sentiment analysis pipeline on the reviews.
    Return tensor that contains all the sentiment scores.

    Args:  df(DataFrame): dataframe to conduct sentiment analysis with

    Returns: torch tensor, one-hot encoded sentiment tensor for the reviews
  '''
  my_device = "cuda" if torch.cuda.is_available() else 'cpu'

  nlp = pipeline(
            "sentiment-analysis",
            model="nlptown/bert-base-multilingual-uncased-sentiment",
            device=my_device,
            truncation=True
        )

  sentiment_tensor = torch.empty((1,5))
  class_map = {"1 star" : 0, "2 stars": 1, "3 stars": 2, "4 stars": 3, "5 stars": 4}
  texts = df["text"].tolist()
  n = len(df)
  for i in tqdm(range(0, n, 128), desc="Sentiment batches"):
    text_batch = texts[i:i+128]
    results = nlp(text_batch)
    result_numeric = []
    for result in results:
      result_numeric.append(class_map[result["label"]])
    result_numeric = torch.tensor(result_numeric)
    one_hot = F.one_hot(result_numeric, num_classes=5)
    sentiment_tensor = torch.cat((sentiment_tensor, torch.squeeze(one_hot.cpu())))

  return sentiment_tensor[1:]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import pandas as pd

def feature_engineer(df, use_sentiment=False):
  '''
    Merge user data and business data with review data, create num_words feature.
    Make tfidf vectors for text, select features to be used, and merge sentiment tensor into data.

    Args:  df(DataFrame): dataframe to use
           use_sentiment (boolean): whether to append sentiment to each review or not

    Returns: sparse.array, np.array, list, list, int. X, y(star label), terms found from tfidf, feature names of data, and number of tfidf terms
  '''
  user_data = pd.read_json("yelp_academic_dataset_user.json", lines=True)
  df = df.merge(user_data, on="user_id", how="left")

  business_data_chunks = pd.read_json("yelp_academic_dataset_business.json", lines=True, chunksize=1000000)
  for business_data_chunk in business_data_chunks:
    df = df.merge(business_data_chunk, on="business_id", how="left")

  print(df.columns)

  df["num_words"] = 0
  for index, row in df.iterrows():
    row["num_words"] = len(row["text"].split())
  corpus = df["text"].astype(str)

  #transform text into tfidf, and build new dataframe with columns =
  tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=200000)
  X = tfidf.fit_transform(corpus)
  tfidf_size = X.shape[1]
  tfidf_terms = np.array(tfidf.get_feature_names_out())

  #creating sparse matrix
  small_df = df[["num_words", "useful_x", "funny_x", "cool_x", "review_count_x", "useful_y", "funny_y", "cool_y",'latitude',
  'longitude',
  'stars_y',
  'review_count_y',
  'is_open',
  'fans',
  'average_stars',
  'compliment_hot',
  'compliment_more',
  'compliment_profile',
  'compliment_cute',
  'compliment_list',
  'compliment_note',
  'compliment_plain',
  'compliment_cool',
  'compliment_funny',
  'compliment_writer',
  'compliment_photos']]
  numpy_df = small_df.to_numpy()

  print(type(numpy_df))
  print(numpy_df.dtype)
  sparse_numpy = sparse.csr_matrix(numpy_df)
  num_feats_names = ["num_words", "useful_x", "funny_x", "cool_x", "review_count_x", "useful_y", "funny_y", "cool_y",'latitude',
  'longitude',
  'stars_y',
  'review_count_y',
  'is_open',
  'fans',
  'average_stars',
  'compliment_hot',
  'compliment_more',
  'compliment_profile',
  'compliment_cute',
  'compliment_list',
  'compliment_note',
  'compliment_plain',
  'compliment_cool',
  'compliment_funny',
  'compliment_writer',
  'compliment_photos']

  final_sparse_array = sparse.hstack((X, sparse_numpy))

  print("done with tfidf")

  #sentiment analysis
  if use_sentiment:
    print("sentiment analysis")
    sent_tensor = sentiment_analysis(df)
    sent_sparse = sparse.csr_matrix(sent_tensor.numpy())
    final_sparse_array = sparse.hstack((final_sparse_array, sent_sparse))

  y = df["stars_x"].to_numpy()
  if use_sentiment:
    feat_names = np.concatenate([tfidf_terms, num_feats_names, ["very negative", "negative", "neutral", "positive", "very positive"]])
  else:
    feat_names = np.concatenate([tfidf_terms, num_feats_names])

  print("DONE WITH FEAT ENGINEER")
  print(final_sparse_array.shape)

  return final_sparse_array, y, tfidf_terms, feat_names, tfidf_size

#Creating Train and Test Sets For Each Ablation

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def make_train_and_test(use_sentiment, use_gibberish, data_with_gibberish_filter_json, data_without_gibberish_filter_json):
  '''
    Create train and test datasets by calling load_data function, and feature_engineer function.
    Call sklearn train_and_test function to split data.

    Args:  use_sentiment(boolean): whether to use sentiment or not
           use_gibberish(boolean): whether to gibberish filter or not
           data_with_gibberish_filter_json(string): filename for data that was filtered
           data_without_gibberish_filter_json(string): filename for data that was not filtered

    Returns: sparse.array, sparse.array, np.array, np.array, list, list, int: train data, test data, train labels, test label, tf-idf terms, and feature names, and number of tfidf terms
  '''
  if use_gibberish:
    review_with_gibberish_filter_df = load_data(data_with_gibberish_filter_json)

    if use_sentiment: #Full approach
      X, y, tfidf_terms, feat_names, tfidf_size  = feature_engineer(review_with_gibberish_filter_df, use_sentiment=True)
    else:  #Ablation 2
      X, y, tfidf_terms, feat_names, tfidf_size = feature_engineer(review_with_gibberish_filter_df, use_sentiment=False)

    del review_with_gibberish_filter_df

  else:
    review_without_gibberish_filter_df = load_data(data_without_gibberish_filter_json)

    if not use_sentiment: #baseline
      X, y, tfidf_terms, feat_names, tfidf_size = feature_engineer(review_without_gibberish_filter_df, use_sentiment=False)
    else: #Ablation 1
      X, y, tfidf_terms, feat_names, tfidf_size = feature_engineer(review_without_gibberish_filter_df, use_sentiment=True)

    del review_without_gibberish_filter_df

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=45, stratify=y)

  return X_train, X_test, y_train, y_test, tfidf_terms, feat_names, tfidf_size


Baseline Approach

In [ ]:
#create datasets
X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline, tfidf_terms_baseline, feat_names_baseline, tfidf_size_baseline = make_train_and_test(False,
                                                                                                                                      False,
                                                                                                                                      "data_with_gibberish_filter.json",
                                                                                                                                      "data_without_gibberish_filter.json") #baseline
print(X_train_baseline.shape)
print(X_test_baseline.shape)
print(y_train_baseline.shape)
print(y_test_baseline.shape)
print(tfidf_size_baseline)

Ablation 1 (Sentiment Analysis + No Gibberish Filtering)

In [ ]:
X_train_ablation1, X_test_ablation1, y_train_ablation1, y_test_ablation1, tfidf_terms_ablation1, feat_names_ablation1, tfidf_size_ablation1 = make_train_and_test(True,
                                                                                                                                            False,
                                                                                                                                            "data_with_gibberish_filter.json",
                                                                                                                                            "data_without_gibberish_filter.json") #ablation 1
print(X_train_ablation1.shape)
print(X_test_ablation1.shape)
print(y_train_ablation1.shape)
print(y_test_ablation1.shape)
print(tfidf_size_ablation1)

Ablation 2 (No Sentiment Analysis + Gibberish Filtering)

In [ ]:
X_train_ablation2, X_test_ablation2, y_train_ablation2, y_test_ablation2, tfidf_terms_ablation2, feat_names_ablation2, tfidf_size_ablation2 = make_train_and_test(False,
                                                                                                                                            True,
                                                                                                                                            "data_with_gibberish_filter.json",
                                                                                                                                            "data_without_gibberish_filter.json") #ablation 2
print(X_train_ablation2.shape)
print(X_test_ablation2.shape)
print(y_train_ablation2.shape)
print(y_test_ablation2.shape)
print(tfidf_size_ablation2)

Full Approach (Sentiment Analysis + Gibberish Filtering)

In [ ]:
X_train_full, X_test_full, y_train_full, y_test_full, tfidf_terms_full, feat_names_full, tfidf_size_full = make_train_and_test(True,
                                                                                                              True,
                                                                                                              "data_with_gibberish_filter.json",
                                                                                                              "data_without_gibberish_filter.json") #full approach
print(X_train_full.shape)
print(X_test_full.shape)
print(y_train_full.shape)
print(y_test_full.shape)
print(tfidf_size_full)

In [ ]:
def save_splits(X_train, X_test, y_train, y_test, tfidf_terms, feat_names, experiment_name):
  '''
    Save X_train, y_train, X_test, y_test, tfidf_terms, and feat_names from a certain experiment.
    Avoid having to process data every single run.

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data
      tfidf_size(int): number of tfidf terms
  '''
  sparse.save_npz(f"X_train_{experiment_name}.npz", X_train)
  sparse.save_npz(f"X_test_{experiment_name}.npz", X_test)
  np.save(f"y_train_{experiment_name}.npy", y_train)
  np.save(f"y_test_{experiment_name}.npy", y_test)
  np.save(f"tfidf_terms_{experiment_name}.npy", tfidf_terms)
  np.save(f"feat_names_{experiment_name}.npy", feat_names)




In [ ]:
save_splits(X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline, tfidf_terms_baseline, feat_names_baseline, "baseline")

In [ ]:
save_splits(X_train_ablation1, X_test_ablation1, y_train_ablation1, y_test_ablation1, tfidf_terms_ablation1, feat_names_ablation1, "ablation1")

In [ ]:
save_splits(X_train_ablation2, X_test_ablation2, y_train_ablation2, y_test_ablation2, tfidf_terms_ablation2, feat_names_ablation2, "ablation2")

In [ ]:
save_splits(X_train_full, X_test_full, y_train_full, y_test_full, tfidf_terms_full, feat_names_full, "full")

Load Saved Matrices

In [ ]:
X_train_baseline = sparse.load_npz("X_train_baseline.npz")
X_test_baseline = sparse.load_npz("X_test_baseline.npz")
y_train_baseline = np.load("y_train_baseline.npy")
y_test_baseline = np.load("y_test_baseline.npy")
tfidf_terms_baseline = np.load("tfidf_terms_baseline.npy", allow_pickle=True)
feat_names_baseline = np.load("feat_names_baseline.npy", allow_pickle=True)
tfidf_size_baseline = 200000 # change as needed


In [ ]:
X_train_ablation1 = sparse.load_npz("X_train_ablation1.npz")
X_test_ablation1 = sparse.load_npz("X_test_ablation1.npz")
y_train_ablation1 = np.load("y_train_ablation1.npy")
y_test_ablation1 = np.load("y_test_ablation1.npy")
tfidf_terms_ablation1 = np.load("tfidf_terms_ablation1.npy", allow_pickle=True)
feat_names_ablation1 = np.load("feat_names_ablation1.npy", allow_pickle=True)
tfidf_size_ablation1 = 200000 # change as needed

In [ ]:
X_train_ablation2 = sparse.load_npz("X_train_ablation2.npz")
X_test_ablation2 = sparse.load_npz("X_test_ablation2.npz")
y_train_ablation2 = np.load("y_train_ablation2.npy")
y_test_ablation2 = np.load("y_test_ablation2.npy")
tfidf_terms_ablation2 = np.load("tfidf_terms_ablation2.npy", allow_pickle=True)
feat_names_ablation2 = np.load("feat_names_ablation2.npy", allow_pickle=True)
tfidf_size_ablation2 = 200000 # change as needed

In [ ]:
X_train_full = sparse.load_npz("X_train_full.npz")
X_test_full = sparse.load_npz("X_test_full.npz")
y_train_full = np.load("y_train_full.npy")
y_test_full = np.load("y_test_full.npy")
tfidf_terms_full = np.load("tfidf_terms_full.npy", allow_pickle=True)
feat_names_full = np.load("feat_names_full.npy", allow_pickle=True)
tfidf_size_full = 200000 # change as needed

In [ ]:
ablations = ["Baseline",
             "Ablation 1",
             "Ablation 2",
             "Full Approach"]

#Models (Ablations + Parameter Hypertuning)

Ridge Model

In [ ]:
def ridge_model(X_train, X_test, y_train, y_test, tfidf_terms, feat_names, solver=None):
  '''
    Call the Ridge model, make report metrics, and find most important features in model.

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data
      solver(string): if provided, use this solver for Ridge classifier

    Return:
      sklearn.base.BaseEstimator, int, int: trained ridge model, accuracy and macro-f1 scores
  '''
  if solver:
    ridge = RidgeClassifier(solver=solver)
  else:
    ridge = RidgeClassifier()

  ridge.fit(X_train, y_train)
  y_pred_ridge = ridge.predict(X_test)


  ridge_accuracy = float(accuracy_score(y_test, y_pred_ridge))
  ridge_macro_f1 = float(f1_score(y_test, y_pred_ridge, average = "macro"))
  print("Ridge accuracy", float(accuracy_score(y_test, y_pred_ridge)))
  print("Ridge macro_f1", float(f1_score(y_test, y_pred_ridge, average = "macro")))
  print("Ridge report", classification_report(y_test, y_pred_ridge, output_dict = True))

  cm_ridge = confusion_matrix(y_test, y_pred_ridge)
  print(pd.DataFrame(cm_ridge))

  # pd.DataFrame(cm_lr).to_csv(os.path.join("./results", "confusion_matrix_logreg.csv"), index = False)

  ridge_coef = zip(feat_names, np.abs(ridge.coef_[0]))
  sorted_ridge_coef = sorted(ridge_coef, key = lambda item: item[1])[::-1][:10]
  print("Ridge sorted coefficeints:")
  print(sorted_ridge_coef)
  return ridge, ridge_accuracy, ridge_macro_f1


In [ ]:
ridge_ablation_acc = []
ridge_macro_f1 = []

In [ ]:
#Run all 4 experiments(Baseline, Ablation 1, Ablation 2, Full Approach)

print(X_train_baseline.shape)
ridge_model_baseline, ridge_accuracy_baseline, ridge_macro_f1_baseline = ridge_model(X_train_baseline,
                                                                                   X_test_baseline,
                                                                                   y_train_baseline,
                                                                                   y_test_baseline,
                                                                                   tfidf_terms_baseline,
                                                                                   feat_names_baseline)
ridge_ablation_acc.append(ridge_accuracy_baseline)
ridge_macro_f1.append(ridge_macro_f1_baseline)

In [ ]:
print(X_train_ablation1.shape)
ridge_model_ablation1, ridge_accuracy_ablation1, ridge_macro_f1_ablation1 = ridge_model(X_train_ablation1,
                                                                                      X_test_ablation1,
                                                                                      y_train_ablation1,
                                                                                      y_test_ablation1,
                                                                                      tfidf_terms_ablation1,
                                                                                      feat_names_ablation1)
ridge_ablation_acc.append(ridge_accuracy_ablation1)
ridge_macro_f1.append(ridge_macro_f1_ablation1)

In [ ]:
ridge_model_ablation2, ridge_accuracy_ablation2, ridge_macro_f1_ablation2 = ridge_model(X_train_ablation2,
                                                                                      X_test_ablation2,
                                                                                      y_train_ablation2,
                                                                                      y_test_ablation2,
                                                                                      tfidf_terms_ablation2,
                                                                                      feat_names_ablation2)
ridge_ablation_acc.append(ridge_accuracy_ablation2)
ridge_macro_f1.append(ridge_macro_f1_ablation2)

In [ ]:
ridge_model_full_approach, ridge_accuracy_full_approach, ridge_macro_f1_full_approach = ridge_model(X_train_full,
                                                                                                  X_test_full,
                                                                                                  y_train_full,
                                                                                                  y_test_full,
                                                                                                  tfidf_terms_full,
                                                                                                  feat_names_full)
ridge_ablation_acc.append(ridge_accuracy_full_approach)
ridge_macro_f1.append(ridge_macro_f1_full_approach)

In [ ]:
# Plotting ablation vs accuracy and ablation vs macro-f1
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ridge_ablation_acc_df = pd.DataFrame({"Ablation": ablations, "Accuracy": ridge_ablation_acc})
plt.figure()
sns.barplot(ridge_ablation_acc_df, x="Ablation", y="Accuracy")
plt.title("Ablation vs Accuracy")
plt.show()
del ridge_ablation_acc_df


# plot of ablation vs macro F1
ridge_ablation_macro_f1_df = pd.DataFrame({"Ablation": ablations, "Macro F1": ridge_macro_f1})
plt.figure()
sns.barplot(ridge_ablation_macro_f1_df, x="Ablation", y="Macro F1")
plt.title("Ablation vs Macro F1")
plt.show()
del ridge_ablation_macro_f1_df

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def ridge_hypertune(X_train, X_test, y_train, y_test, tfidf_terms, feat_names, tfidf_size):
  '''
    Create new model with different solver for parameter study.

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data
      tfidf_size(int): number of features from tfidf
  '''

  ridge_model_full_approach_max_iter, ridge_accuracy_full_approach_max_iter, ridge_macro_f1_full_approach_max_iter = ridge_model(X_train_full,
                                                                                                  X_test_full,
                                                                                                  y_train_full,
                                                                                                  y_test_full,
                                                                                                  tfidf_terms_full,
                                                                                                  feat_names_full, solver="sag")



def find_most_important_terms_per_class(best_ridge_model, feat_names, tfidf_size, tfidf_terms):
  '''
    For the given model find the most important terms in each class as well as the most important terms for entire model.
    Call the Ridge model, make report metrics, and find most important features in model.

    Args:
      best_ridge_model(sklearn.base.BaseEstimator)
  '''
  ridge_coef = zip(feat_names, np.abs(best_ridge_model.coef_[0]))
  sorted_ridge_coef = sorted(ridge_coef, key=lambda item: item[1])[::-1][:10]
  print("RidgeClassifier Sorted Coefficients:")
  print(sorted_ridge_coef)

  # top words for each class
  ridge_coef_tfidf = best_ridge_model.coef_[:, :tfidf_size]
  top_term_amount = 10
  top_tfidf_terms = {}

  for star_class in range(ridge_coef_tfidf.shape[0]):
    term_index = np.argsort(ridge_coef_tfidf[star_class])[::-1][:top_term_amount]
    top_tfidf_terms[star_class + 1] = [str(tfidf_terms[i]) for i in term_index]

  print("RidgeClassifier Top Terms Per Star Rating:")
  print(top_tfidf_terms)

# hypertune best ablation model, find most important terms per class
ridge_hypertune(X_train_full, X_test_full, y_train_full, y_test_full, tfidf_terms_full, feat_names_full, 200000)
find_most_important_terms_per_class(ridge_model_full_approach, feat_names_full, tfidf_size_full, tfidf_terms_full)


Logistic Regression Model

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def log_reg_model(X_train, X_test, y_train, y_test, tfidf_terms, feat_names):
  '''
    Create Logistic Regression model, print main metrics, and find top terms.

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data

    Return:
      sklearn.base.BaseEstimator, int, int: trained lr model, accuracy and macro-f1 scores
  '''
  lr = LogisticRegression()
  lr.fit(X_train, y_train)
  y_pred_lr = lr.predict(X_test)

  lr_accuracy = float(accuracy_score(y_test, y_pred_lr))
  lr_macro_f1 = float(f1_score(y_test, y_pred_lr, average = "macro"))
  print("Logreg accuracy", lr_accuracy)
  print("Logreg macro_f1", lr_macro_f1)
  print(classification_report(y_test, y_pred_lr))

  cm_lr = confusion_matrix(y_test, y_pred_lr)
  print(pd.DataFrame(cm_lr))


  # pd.DataFrame(cm_lr).to_csv(os.path.join("./results", "confusion_matrix_logreg.csv"), index = False)

  lr_coef = zip(feat_names, np.abs(lr.coef_[0]))
  sorted_lr_coef = sorted(lr_coef, key = lambda item: item[1])[::-1][:10]
  print("SVM sorted coefficeints:")
  print(sorted_lr_coef)

  return lr, lr_accuracy, lr_macro_f1

In [ ]:
#Run experiments

lr_ablation_acc = []
lr_macro_f1 = []

log_reg_model_baseline, lr_accuracy_baseline, lr_macro_f1_baseline = log_reg_model(X_train_baseline,
                                                                                   X_test_baseline,
                                                                                   y_train_baseline,
                                                                                   y_test_baseline,
                                                                                   tfidf_terms_baseline,
                                                                                   feat_names_baseline)
lr_ablation_acc.append(lr_accuracy_baseline)
lr_macro_f1.append(lr_macro_f1_baseline)

In [ ]:
log_reg_model_ablation1, lr_accuracy_ablation1, lr_macro_f1_ablation1 = log_reg_model(X_train_ablation1,
                                                                                      X_test_ablation1,
                                                                                      y_train_ablation1,
                                                                                      y_test_ablation1,
                                                                                      tfidf_terms_ablation1,
                                                                                      feat_names_ablation1)
lr_ablation_acc.append(lr_accuracy_ablation1)
lr_macro_f1.append(lr_macro_f1_ablation1)

In [ ]:
log_reg_model_ablation2, lr_accuracy_ablation2, lr_macro_f1_ablation2 = log_reg_model(X_train_ablation2,
                                                                                      X_test_ablation2,
                                                                                      y_train_ablation2,
                                                                                      y_test_ablation2,
                                                                                      tfidf_terms_ablation2,
                                                                                      feat_names_ablation2)
lr_ablation_acc.append(lr_accuracy_ablation2)
lr_macro_f1.append(lr_macro_f1_ablation2)

In [ ]:
log_reg_model_full_approach, lr_accuracy_full_approach, lr_macro_f1_full_approach = log_reg_model(X_train_full,
                                                                                                  X_test_full,
                                                                                                  y_train_full,
                                                                                                  y_test_full,
                                                                                                  tfidf_terms_full,
                                                                                                  feat_names_full)
lr_ablation_acc.append(lr_accuracy_full_approach)
lr_macro_f1.append(lr_macro_f1_full_approach)

In [ ]:
# Plotting ablation vs accuracy and ablation vs macro-f1
import pandas as pd

lr_ablation_acc_df = pd.DataFrame({"Ablation": ablations, "Accuracy": lr_ablation_acc})
plt.figure()
sns.barplot(lr_ablation_acc_df, x="Ablation", y="Accuracy")
plt.title("Ablation vs Accuracy")
plt.show()
del lr_ablation_acc_df


# plot of ablation vs macro F1
lr_ablation_macro_f1_df = pd.DataFrame({"Ablation": ablations, "Macro F1": lr_macro_f1})
plt.figure()
sns.barplot(lr_ablation_macro_f1_df, x="Ablation", y="Macro F1")
plt.title("Ablation vs Macro F1")
plt.show()
del lr_ablation_macro_f1_df



In [ ]:
from sklearn.model_selection import ParameterGrid, GridSearchCV

def log_reg_hypertune(best_ablation_log_reg_model, X_train, X_test, y_train, y_test, tfidf_terms, feat_names, tfidf_size):
  '''
    Utilize GridSearchCV to experiment with C value for Log Reg model.

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data
      tfidf_size(int): number of features from tfidf
  '''
  lr_param_grid = [
      { #'penalty': ['l1', 'l2'],
       'C': [0.01, 0.1, 1, 10, 100],
      #  'solver': ['saga'],
      #  'max_iter': [1000, 2000]},
      #   {'penalty': ['elasticnet'],
      #    'l1_ratio': [0.25, 0.5, 0.75],
      #    'C': [0.001, 0.01, 0.1, 1, 10],
      #    'solver': ['saga'],
      #    'max_iter': [1000, 2000]
       }
    ]

  lr_grid_search = GridSearchCV(best_ablation_log_reg_model,
                                     param_grid=lr_param_grid,
                                     cv=2,
                                     verbose=True,
                                     n_jobs=-1)

  lr_grid_search.fit(X_train, y_train)

  print("Best Paramaters: ", lr_grid_search.best_params_)

  lr_best_param_model = lr_grid_search.best_estimator_

  y_pred_best_param_lr = lr_best_param_model.predict(X_test)

  print("Logreg accuracy", float(accuracy_score(y_test, y_pred_best_param_lr)))
  print("Logreg macro_f1", float(f1_score(y_test, y_pred_best_param_lr, average = "macro")))
  print("Logreg report", classification_report(y_test, y_pred_best_param_lr, output_dict = True))

  cm_lr_best_param = confusion_matrix(y_test, y_pred_best_param_lr)
  print(pd.DataFrame(cm_lr_best_param))


def find_most_important_terms_per_class_lr(best_lr_model, feat_names, tfidf_size, tfidf_terms):
  '''
    Find the most important terms per class for the given model as well as the most important terms for entire model

    Args:
      best_ridge_model(sklearn.base.BaseEstimator)
  '''
  lr_coef = zip(feat_names, np.abs(best_lr_model.coef_[0]))
  sorted_lr_coef = sorted(lr_coef, key = lambda item: item[1])[::-1][:10]
  print("Logistic Regression Sorted Coefficients:")
  print(sorted_lr_coef)

  # top words for each class
  lr_coef_tfidf = best_lr_model.coef_[:, :tfidf_size]
  top_term_amount = 10
  top_tfidf_terms = {}

  for star_class in range(lr_coef_tfidf.shape[0]):
    term_index = np.argsort(lr_coef_tfidf[star_class])[::-1][:top_term_amount]
    top_tfidf_terms[star_class + 1] = [str(tfidf_terms[i]) for i in term_index]

  print("Logistic Regression Top Terms Per Star Rating:")
  print(top_tfidf_terms)

# hypertune best ablation model
log_reg_hypertune(log_reg_model_full_approach, X_train_full, X_test_full, y_train_full, y_test_full, tfidf_terms_full, feat_names_full, 200000)
find_most_important_terms_per_class_lr(log_reg_model_full_approach, feat_names_full, tfidf_size_full, tfidf_terms_full)


Linear SVC Model

In [ ]:
def linear_svc_model(X_train, X_test, y_train, y_test, tfidf_terms, feat_names, max_iter=None):
  '''
    Call the Linear SVC model, make report metrics, and find most important features in model.

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data

    Return:
      sklearn.base.BaseEstimator, int, int: trained svc model, accuracy and macro-f1 scores
  '''
     # SVM
  if max_iter:
    svm = LinearSVC(max_iter = max_iter)
  else:
    svm = LinearSVC()
  svm.fit(X_train, y_train)
  y_pred_svm = svm.predict(X_test)

  svm_accuracy = float(accuracy_score(y_test, y_pred_svm))
  svm_macro_f1 = float(f1_score(y_test, y_pred_svm, average = "macro"))
  print("SVM accuracy", float(accuracy_score(y_test, y_pred_svm)))
  print("SVM macro_f1", float(f1_score(y_test, y_pred_svm, average = "macro")))
  print("SVM report", classification_report(y_test, y_pred_svm, output_dict = True))

  cm_svm = confusion_matrix(y_test, y_pred_svm)
  print(pd.DataFrame(cm_svm))

  # pd.DataFrame(cm_lr).to_csv(os.path.join("./results", "confusion_matrix_logreg.csv"), index = False)

  svm_coef = zip(feat_names, np.abs(svm.coef_[0]))
  sorted_svm_coef = sorted(svm_coef, key = lambda item: item[1])[::-1][:10]
  print("SVM sorted coefficeints:")
  print(sorted_svm_coef)

  return svm, svm_accuracy, svm_macro_f1


In [ ]:
# Run experiments
svc_ablation_acc = []
svc_macro_f1 = []

svc_model_baseline, svc_accuracy_baseline, svc_macro_f1_baseline = linear_svc_model(X_train_baseline,
                                                                                   X_test_baseline,
                                                                                   y_train_baseline,
                                                                                   y_test_baseline,
                                                                                   tfidf_terms_baseline,
                                                                                   feat_names_baseline)
svc_ablation_acc.append(svc_accuracy_baseline)
svc_macro_f1.append(svc_macro_f1_baseline)

In [ ]:
svc_model_ablation1, svc_accuracy_ablation1, svc_macro_f1_ablation1 = linear_svc_model(X_train_ablation1,
                                                                                      X_test_ablation1,
                                                                                      y_train_ablation1,
                                                                                      y_test_ablation1,
                                                                                      tfidf_terms_ablation1,
                                                                                      feat_names_ablation1)
svc_ablation_acc.append(svc_accuracy_ablation1)
svc_macro_f1.append(svc_macro_f1_ablation1)

In [ ]:
svc_model_ablation2, svc_accuracy_ablation2, svc_macro_f1_ablation2 = linear_svc_model(X_train_ablation2,
                                                                                      X_test_ablation2,
                                                                                      y_train_ablation2,
                                                                                      y_test_ablation2,
                                                                                      tfidf_terms_ablation2,
                                                                                      feat_names_ablation2)
svc_ablation_acc.append(svc_accuracy_ablation2)
svc_macro_f1.append(svc_macro_f1_ablation2)

In [ ]:
svc_model_full_approach, svc_accuracy_full_approach, svc_macro_f1_full_approach = linear_svc_model(X_train_full,
                                                                                                  X_test_full,
                                                                                                  y_train_full,
                                                                                                  y_test_full,
                                                                                                  tfidf_terms_full,
                                                                                                  feat_names_full)
svc_ablation_acc.append(svc_accuracy_full_approach)
svc_macro_f1.append(svc_macro_f1_full_approach)

In [ ]:
# Plotting ablation vs accuracy and ablation vs macro-f1
import pandas as pd

svc_ablation_acc_df = pd.DataFrame({"Ablation": ablations, "Accuracy": svc_ablation_acc})
plt.figure()
sns.barplot(svc_ablation_acc_df, x="Ablation", y="Accuracy")
plt.title("Ablation vs Accuracy")
plt.show()
del svc_ablation_acc_df


# plot of ablation vs macro F1
svc_ablation_macro_f1_df = pd.DataFrame({"Ablation": ablations, "Macro F1": svc_macro_f1})
plt.figure()
sns.barplot(svc_ablation_macro_f1_df, x="Ablation", y="Macro F1")
plt.title("Ablation vs Macro F1")
plt.show()
del svc_ablation_macro_f1_df

In [ ]:
from sklearn.model_selection import ParameterGrid, GridSearchCV

def svc_hypertune(X_train, X_test, y_train, y_test, tfidf_terms, feat_names, tfidf_size):
  '''
    Create new model with new value for max_iter for parameter study

    Args:
      X_train(sparse.array): train data
      X_test(sparse.array): test data
      y_train(np.array): train labels
      y_test(np.array): test labels
      tfidf_terms(list): terms from tfidf
      feat_names(list): feature names for data
      tfidf_size(int): number of features from tfidf
  '''

  svc_model_full_approach_max_iter, svc_accuracy_full_approach_max_iter, svc_macro_f1_full_approach_max_iter = linear_svc_model(X_train_full,
                                                                                                  X_test_full,
                                                                                                  y_train_full,
                                                                                                  y_test_full,
                                                                                                  tfidf_terms_full,
                                                                                                  feat_names_full,
                                                                                                  2000)

def find_most_important_terms_per_class_svc(best_svc_model, feat_names, tfidf_size, tfidf_terms):
  '''
    Find most important terms for model and for the star classes.

    Args:
      best_ridge_model(sklearn.base.BaseEstimator)
  '''
  svc_coef = zip(feat_names, np.abs(best_svc_model.coef_[0]))
  sorted_svc_coef = sorted(svc_coef, key = lambda item: item[1])[::-1][:10]
  print("Support Vector Machine Sorted Coefficients:")
  print(sorted_svc_coef)

  # top words for each class
  svc_coef_tfidf = best_svc_model.coef_[:, :tfidf_size]
  top_term_amount = 10
  top_tfidf_terms = {}

  for star_class in range(svc_coef_tfidf.shape[0]):
    term_index = np.argsort(svc_coef_tfidf[star_class])[::-1][:top_term_amount]
    top_tfidf_terms[star_class + 1] = [str(tfidf_terms[i]) for i in term_index]

  print("Support Vector Machine Top Terms Per Star Rating:")
  print(top_tfidf_terms)

# hypertune best ablation model
svc_hypertune(X_train_full, X_test_full, y_train_full, y_test_full, tfidf_terms_full, feat_names_full, 200000)
find_most_important_terms_per_class_svc(svc_model_full_approach, feat_names_full, tfidf_size_full, tfidf_terms_full)